# 📊 PTA Treasurer Report Generator — v4
### File-name driven · Auto-detects month · Updates correct column · GitHub integration
---
**Monthly workflow:**
1. Drop files in input folders (`quickbooks_february_2026.csv`, `givebacks_february.csv`, bank PDF)
2. Run Cells 0–10 to generate the Excel report
3. Run Cell 11 to push code changes to GitHub

**File naming:**
- QuickBooks → `quickbooks_<month>_<year>.csv`  e.g. `quickbooks_february_2026.csv`
- Givebacks  → `givebacks_<month>.csv`           e.g. `givebacks_february.csv`
- Bank PDF   → any name (month detected from content)


## Cell 0 — Install Dependencies
*Run once.*

In [1]:
import subprocess, sys, platform
print(f'Python: {sys.version}')
print(f'OS: {platform.system()}')

print('\nInstalling Python packages...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'openpyxl', 'pdfplumber', 'playwright', 'python-dotenv', '-q'])
print('  Done')

print('\nInstalling Playwright browser...')
try:
    r = subprocess.run(
        [sys.executable, '-m', 'playwright', 'install', 'chromium', '--with-deps'],
        capture_output=True, text=True)
    if r.returncode != 0:
        r2 = subprocess.run(
            [sys.executable, '-m', 'playwright', 'install', 'chromium'],
            capture_output=True, text=True)
        if r2.returncode != 0:
            print('  WARNING: Playwright install failed.')
            print('  Skip Cell 2 and upload your Givebacks CSV manually.')
        else:
            print('  Done')
    else:
        print('  Done')
except Exception as e:
    print(f'  WARNING: {e}')
    print('  Skip Cell 2 and upload your Givebacks CSV manually.')

print('\nAll dependencies ready.')


Python: 3.8.5 (default, Sep  4 2020, 02:22:02) 
[Clang 10.0.0 ]
OS: Darwin

Installing Python packages...
  Done

Installing Playwright browser...
  Done

All dependencies ready.


## Cell 1 — Configuration
*Set credentials. Month is auto-detected from filenames.*

In [2]:
from pathlib import Path
from datetime import datetime
import os, re, json, csv
from dotenv import load_dotenv

load_dotenv()

# Organisation name
ORG_NAME = os.getenv('ORG_NAME', 'Setauket School PTA')
INPUT_MONTH = "April"
# Input / output folders
GB_FOLDER   = Path('input/givebacks')/INPUT_MONTH
QB_FOLDER   = Path('input/quickbooks')
BANK_FOLDER = Path('input/bank')
HISTORY_DIR = Path('data/history')

# Givebacks credentials (set here or in .env)
GIVEBACKS_EMAIL    = os.getenv('GIVEBACKS_EMAIL',    '')
GIVEBACKS_PASSWORD = os.getenv('GIVEBACKS_PASSWORD', '')
GIVEBACKS_URL      = os.getenv('GIVEBACKS_URL',      'https://app.mygivebacks.com')

# GitHub settings (set here or in .env)
GITHUB_REMOTE = os.getenv('GITHUB_REMOTE', 'origin')   # remote name
GITHUB_BRANCH = os.getenv('GITHUB_BRANCH', 'main')     # branch to push to

# Fiscal year: July = index 0, June = index 11
FISCAL_MONTHS      = ['JULY','AUG','SEPT','OCT','NOV','DEC','JAN','FEB','MAR','APR','MAY','JUNE']
FISCAL_START_MONTH = 7   # July

MONTH_NAMES = {
    'january':0,'february':1,'march':2,'april':3,'may':4,'june':5,
    'july':6,'august':7,'september':8,'october':9,'november':10,'december':11,
    'jan':0,'feb':1,'mar':2,'apr':3,'jun':5,
    'jul':6,'aug':7,'sep':8,'oct':9,'nov':10,'dec':11
}

def calendar_to_fiscal(cal_month_num):
    return (cal_month_num - FISCAL_START_MONTH) % 12

def month_name_to_fiscal_index(month_name):
    cal_idx = MONTH_NAMES.get(month_name.lower())
    if cal_idx is None: return None
    return calendar_to_fiscal(cal_idx + 1)

for folder in [GB_FOLDER, QB_FOLDER, BANK_FOLDER, Path('output'), HISTORY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f'Organisation : {ORG_NAME}')
print(f'Fiscal year  : July (index 0) -> June (index 11)')
print(f'Givebacks    : {"credentials set" if GIVEBACKS_EMAIL else "no credentials - manual upload"}')
print(f'GitHub       : remote={GITHUB_REMOTE}  branch={GITHUB_BRANCH}')


Organisation : Setauket School PTA
Fiscal year  : July (index 0) -> June (index 11)
Givebacks    : credentials set
GitHub       : remote=origin  branch=main


## Cell 2 — Auto-Download Givebacks *(optional)*
Skip if uploading manually. Requires credentials in Cell 1 or `.env`.

In [6]:
import asyncio, calendar as cal_lib

async def download_givebacks(month_label, email, password, url, dest_folder):
    from playwright.async_api import async_playwright, TimeoutError as PWTimeout
    safe     = month_label.replace(' ', '_').lower()
    month_short = month_label.split()[0].lower()
    existing = list(dest_folder.glob(f'givebacks_{month_short}*.csv'))
    if existing:
        print(f'Already downloaded: {existing[0].name}  (delete to re-download)')
        return existing[0]
    print(f'Opening browser for {month_label}...')
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True,
                                          args=['--no-sandbox','--disable-dev-shm-usage'])
        ctx  = await browser.new_context(accept_downloads=True)
        page = await ctx.new_page()
        try:
            await page.goto(f'{url}/login', timeout=30000)
            await page.wait_for_load_state('networkidle')
            await page.fill('input[type="email"]', email)
            await page.fill('input[type="password"]', password)
            await page.click('button[type="submit"]')
            await page.wait_for_load_state('networkidle')
            if await page.locator('input[name="code"]').count() > 0:
                code_val = input('2FA code: ').strip()
                await page.fill('input[name="code"]', code_val)
                await page.click('button[type="submit"]')
                await page.wait_for_load_state('networkidle')
            if '/login' in page.url:
                raise Exception('Login failed - check credentials')
            print('  Logged in')
            try:
                await page.goto(f'{url}/reports/transactions', timeout=15000)
                await page.wait_for_load_state('networkidle')
            except PWTimeout:
                await page.click('a:has-text("Reports")')
                await page.click('a:has-text("Transactions")')
                await page.wait_for_load_state('networkidle')
            target   = datetime.strptime(month_label, '%B %Y')
            last_day = cal_lib.monthrange(target.year, target.month)[1]
            date_inputs = await page.locator('input[type="date"]').all()
            if len(date_inputs) >= 2:
                await date_inputs[0].fill(target.replace(day=1).strftime('%Y-%m-%d'))
                await date_inputs[1].fill(target.replace(day=last_day).strftime('%Y-%m-%d'))
                btn = page.locator('button:has-text("Apply"), button:has-text("Filter")')
                if await btn.count() > 0:
                    await btn.first.click()
                    await page.wait_for_load_state('networkidle')
            async with page.expect_download(timeout=30000) as dl_info:
                await page.locator('button:has-text("Export"), a:has-text("CSV")').first.click()
            dl   = await dl_info.value
            dest = dest_folder / f'givebacks_{month_short}_{datetime.today().strftime("%Y%m%d")}.csv'
            await dl.save_as(dest)
            print(f'  Saved: {dest.name}')
            return dest
        finally:
            await ctx.close()
            await browser.close()

if not GIVEBACKS_EMAIL or not GIVEBACKS_PASSWORD:
    print('No credentials found.')
    print('Place your Givebacks CSV in input/givebacks/ named: givebacks_february.csv')
else:
    qb_files = sorted(QB_FOLDER.glob('quickbooks_*.csv'))
    if qb_files:
        m = re.search(r'quickbooks_([a-z]+)_?(\d{4})?\.csv', qb_files[-1].name.lower())
        if m:
            month_str = m.group(1).capitalize()
            year_str  = m.group(2) or str(datetime.today().year)
            label     = f'{month_str} {year_str}'
            print(f'Detected month from QB file: {label}')
            try:
                await download_givebacks(label, GIVEBACKS_EMAIL, GIVEBACKS_PASSWORD,
                                         GIVEBACKS_URL, GB_FOLDER)
            except Exception as e:
                print(f'Download failed: {e}')
                print('Upload Givebacks CSV manually to input/givebacks/')
    else:
        print('No QuickBooks file found yet.')


Detected month from QB file: April 2026
Already downloaded: givebacks_april_1.csv  (delete to re-download)


## Cell 3 — Detect Month From Filenames
Reads filenames and PDF content to determine which fiscal month to update.


In [7]:
GB_FOLDER

PosixPath('input/givebacks/April')

In [9]:
import pdfplumber
def detect_month_from_filename(filepath):
    name  = filepath.stem.lower()
    parts = re.split(r'[_\-\s]+', name)
    month_str = None; year_str = None; fiscal_idx = None
    for part in parts:
        if part in MONTH_NAMES and month_str is None:
            month_str  = part
            fiscal_idx = month_name_to_fiscal_index(part)
        if re.match(r'^20\d{2}$', part) and year_str is None:
            year_str = part
    if month_str is None: return None, None
    year_str    = year_str or str(datetime.today().year)
    month_label = f'{month_str.capitalize()} {year_str}'
    return month_label, fiscal_idx

def detect_month_from_pdf(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text() or ''
        m = re.search(
            r'(January|February|March|April|May|June|July|August|'
            r'September|October|November|December)\s+(\d{1,2}),\s+(\d{4})'
            r'\s+through', text, re.IGNORECASE)
        if m:
            return f'{m.group(1)} {m.group(3)}', month_name_to_fiscal_index(m.group(1))
    except Exception as e:
        print(f'  Could not read PDF: {e}')
    return None, None

# QuickBooks
qb_files = sorted(QB_FOLDER.glob('quickbooks_*.csv'))
if not qb_files:
    raise FileNotFoundError( f'No QuickBooks file in {QB_FOLDER}\nName it: quickbooks_february_2026.csv')
QB_FILE = qb_files[-1]
QB_MONTH_LABEL, QB_FISCAL_IDX = detect_month_from_filename(QB_FILE)
if QB_MONTH_LABEL is None:
    raise ValueError(f'Could not detect month from: {QB_FILE.name}\nExpected: quickbooks_february_2026.csv')

print(f'QuickBooks : {QB_FILE.name}  ->  {QB_MONTH_LABEL}  (fiscal index {QB_FISCAL_IDX}: {FISCAL_MONTHS[QB_FISCAL_IDX]})')

# Givebacks
#gb_files = sorted(GB_FOLDER.glob('givebacks_*.csv'))
gb_files = sorted(f for f in GB_FOLDER.rglob('*.csv') if 'givebacks' in f.name.lower())
if not gb_files:
    raise FileNotFoundError(
        f'No Givebacks files in {GB_FOLDER}\nName them: givebacks_february.csv')
GB_FILE_INFO = []
print(f'\nGivebacks :')
for f in gb_files:
    lbl, idx = detect_month_from_filename(f)
    if lbl:
        print(f'  {f.name}  ->  {lbl}  (fiscal index {idx})')
        GB_FILE_INFO.append((f, lbl, idx))
    else:
        print(f'  {f.name}  ->  WARNING: could not detect month, skipping')

# Bank PDF
pdf_files = sorted(BANK_FOLDER.glob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'No PDF in {BANK_FOLDER}')
BANK_FILE = pdf_files[-1]
BANK_MONTH_LABEL, BANK_FISCAL_IDX = detect_month_from_pdf(BANK_FILE)
print(f'\nBank PDF  : {BANK_FILE.name}  ->  ', end='')
if BANK_MONTH_LABEL:
    print(f'{BANK_MONTH_LABEL}  (fiscal index {BANK_FISCAL_IDX})')
else:
    BANK_MONTH_LABEL, BANK_FISCAL_IDX = QB_MONTH_LABEL, QB_FISCAL_IDX
    print(f'month not detected - using QB month: {QB_MONTH_LABEL}')

MONTH_LABEL = QB_MONTH_LABEL
FISCAL_IDX  = QB_FISCAL_IDX
safe_month  = MONTH_LABEL.replace(' ', '_')
OUTPUT_FILE = Path(f'output/Treasurer_Report_{safe_month}.xlsx')

print(f" {'='*55}")
print(f'  Report month : {MONTH_LABEL}')
print(f'  Fiscal index : {FISCAL_IDX}  -> column {FISCAL_MONTHS[FISCAL_IDX]} in budget sheets')
print(f'  Output       : {OUTPUT_FILE}')
print(f"{'='*55}")


QuickBooks : quickbooks_april_2026.csv  ->  April 2026  (fiscal index 9: APR)

Givebacks :
  givebacks_april_1.csv  ->  April 2026  (fiscal index 9)
  givebacks_april_2.csv  ->  April 2026  (fiscal index 9)
  givebacks_april_3.csv  ->  April 2026  (fiscal index 9)
  givebacks_april_4.csv  ->  April 2026  (fiscal index 9)

Bank PDF  : Chase_april_2026.pdf  ->  month not detected - using QB month: April 2026
  Report month : April 2026
  Fiscal index : 9  -> column APR in budget sheets
  Output       : output/Treasurer_Report_April_2026.xlsx


In [10]:
print('QB_FILE exists:  ', QB_FILE.exists())
print('GB files found:  ', list(GB_FOLDER.glob('*.csv')))
print('Bank files found:', list(BANK_FOLDER.glob('*.pdf')))

QB_FILE exists:   True
GB files found:   [PosixPath('input/givebacks/April/givebacks_april_1.csv'), PosixPath('input/givebacks/April/givebacks_april_2.csv'), PosixPath('input/givebacks/April/givebacks_april_3.csv'), PosixPath('input/givebacks/April/givebacks_april_4.csv')]
Bank files found: [PosixPath('input/bank/Chase_april_2026.pdf')]


## Cell 4 — Parse Files & Save to History

In [14]:
from collections import defaultdict

HISTORY_DIR.mkdir(parents=True, exist_ok=True)
safe_month = MONTH_LABEL.replace(' ', '_')

def parse_quickbooks(path):
    data = {'period':'','income':{},'income_total':0.0,'expenses':{},'expense_total':0.0,'net_income':0.0}
    with open(path, newline='', encoding='utf-8-sig') as f:
        lines = list(csv.reader(f))
    if len(lines) > 2: data['period'] = lines[2][0].strip()
    section = None
    for row in lines:
        if not row or not row[0].strip(): continue
        label = row[0].strip()
        try:    val = float(str(row[1]).replace('$','').replace(',','').strip()) if len(row)>1 else 0.0
        except: val = 0.0
        if label == 'Income':                          section = 'income'
        elif label == 'Expenses':                      section = 'expenses'
        elif label.startswith('Total for Income'):     data['income_total'] = val
        elif label.startswith('Total for Expenses'):   data['expense_total'] = val
        elif label.startswith('Net Income'):           data['net_income'] = val
        elif any(label.startswith(x) for x in ['Total for','Gross Profit','Net Operating','Net Other']): continue
        elif section == 'income'   and val != 0.0:     data['income'][label] = val
        elif section == 'expenses' and val != 0.0:     data['expenses'][label] = val
    return data

def parse_givebacks_files(file_info_list):
    merged = {}
    for fpath, lbl, idx in file_info_list:
        with open(fpath, newline='', encoding='utf-8-sig') as f:
            for r in csv.DictReader(f):
                item = r.get('Item','').strip()
                if not item: continue
                try:    amt = float(str(r.get('Total','0')).replace('$','').replace(',',''))
                except: amt = 0.0
                try:    cnt = int(r.get('No. of Transactions','0').strip() or 0)
                except: cnt = 0
                if item in merged:
                    merged[item]['count'] += cnt
                    merged[item]['total'] += amt
                    merged[item]['sources'].add(fpath.name)
                else:
                    merged[item] = {'item':item,'category':r.get('Categories','').strip(),
                                    'count':cnt,'total':amt,'sources':{fpath.name}}
    rows = list(merged.values())
    for r in rows: r['source_file'] = ', '.join(sorted(r['sources'])); del r['sources']
    return rows

def parse_chase_pdf(path):
    bank = {'period':'','account':'','beginning_balance':0.0,'ending_balance':0.0,
            'total_deposits':0.0,'total_checks':0.0,'total_fees':0.0,
            'total_withdrawals':0.0,                                          # ← new
            'deposits':[],'checks':[],'fees':[],'withdrawals':[],             # ← new
            'daily_balances':{},'source_file':path.name}
    with pdfplumber.open(path) as pdf:
        text = chr(10).join(p.extract_text() or '' for p in pdf.pages)
    def grab(pat):
        m = re.search(pat, text)
        try: return float(m.group(1).replace(',','')) if m else 0.0
        except: return 0.0
    m = re.search(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d+,\s+\d{4}\s+through\s+\S+\s+\d+,\s+\d{4}', text)
    if m: bank['period'] = m.group(0)
    m = re.search(r'Account Number:\s+([\d]+)', text)
    if m: bank['account'] = m.group(1)
    bank['beginning_balance'] = grab(r'Beginning Balance\s+\$?([\d,]+\.\d{2})')
    bank['ending_balance']    = grab(r'Ending Balance\s+\d+\s+\$?([\d,]+\.\d{2})')
    bank['total_deposits']    = grab(r'Total Deposits and Additions\s+\$?([\d,]+\.\d{2})')
    bank['total_checks']      = grab(r'Total Checks Paid\s+\$?([\d,]+\.\d{2})')
    bank['total_fees']        = grab(r'Total Fees\s+\$?([\d,]+\.\d{2})')
    bank['total_withdrawals'] = grab(r'Total (Electronic Withdrawals?|Withdrawals? and Transfers?|Withdrawals?)\s+\$?([\d,]+\.\d{2})')  # ← new

    dep = re.search(r'DEPOSITS AND ADDITIONS(.*?)CHECKS PAID', text, re.DOTALL)
    if dep:
        for m in re.finditer(r'(\d{2}/\d{2})\s+(.+?)\s+\$?([\d,]+\.\d{2})', dep.group(1)):
            bank['deposits'].append({'date':m.group(1),'description':m.group(2).strip(),'amount':float(m.group(3).replace(',',''))})

    for m in re.finditer(r'(\d{4})\s+\^?\s+(\d{2}/\d{2})\s+\$?([\d,]+\.\d{2})', text):
        bank['checks'].append({'check_no':m.group(1),'date':m.group(2),'amount':float(m.group(3).replace(',',''))})

    fee = re.search(r'FEES(.*?)DAILY ENDING BALANCE', text, re.DOTALL)
    if fee:
        for m in re.finditer(r'(\d{2}/\d{2})\s+(.+?)\s+\$?([\d,]+\.\d{2})', fee.group(1)):
            bank['fees'].append({'date':m.group(1),'description':m.group(2).strip(),'amount':float(m.group(3).replace(',',''))})

    # ── Withdrawals section ────────────────────────────────────────────────── new
    wd = re.search(r'(OTHER WITHDRAWALS?|WITHDRAWALS? AND TRANSFERS?|WITHDRAWALS?)(.*?)(FEES|DAILY ENDING BALANCE|CHECKS PAID)', text, re.DOTALL | re.IGNORECASE)
    if wd:
        for m in re.finditer(r'(\d{2}/\d{2})\s+(.+?)\s+([\d,]+\.\d{2})', wd.group(2)):
            bank['withdrawals'].append({
                'date':        m.group(1),
                'description': m.group(2).strip(),
                'amount':      float(m.group(3).replace(',',''))
            })
        # If total_withdrawals wasn't caught by grab(), sum from line items
        if bank['total_withdrawals'] == 0.0 and bank['withdrawals']:
            bank['total_withdrawals'] = sum(w['amount'] for w in bank['withdrawals'])
    # ──────────────────────────────────────────────────────────────────────────

    bal = re.search(r'DAILY ENDING BALANCE(.*)$', text, re.DOTALL)
    if bal:
        for m in re.finditer(r'(\d{2}/\d{2})\s+([\d,]+\.\d{2})', bal.group(1)):
            bank['daily_balances'][m.group(1)] = float(m.group(2).replace(',',''))
    return bank

print(f'QuickBooks : income=${qb["income_total"]:,.2f}  expenses=${qb["expense_total"]:,.2f}  net=${qb["net_income"]:,.2f}')
print(f'Givebacks  : {len(givebacks)} items  total=${sum(g["total"] for g in givebacks):,.2f}')
print(f'Bank       : beginning=${bank["beginning_balance"]:,.2f}  ending=${bank["ending_balance"]:,.2f}')

# Save to history

try:
    safe_month = MONTH_LABEL.replace(' ', '_')
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)
    hist_entry = {
        'month_label':     MONTH_LABEL,
        'fiscal_index':    FISCAL_IDX,
        'income':          qb['income'],
        'expenses':        qb['expenses'],
        'income_total':    qb['income_total'],
        'expense_total':   qb['expense_total'],
        'net_income':      qb['net_income'],
        'givebacks_total': sum(g['total'] for g in givebacks),
        'generated_at':    datetime.today().isoformat(),
    }
    hist_path = HISTORY_DIR / f'{safe_month}.json'
    hist_path.write_text(json.dumps(hist_entry, indent=2))
    print(f'\nHistory saved -> {hist_path}')
except Exception as e:
    print(f'\nWARNING: Could not save history: {e}')
    print('Report generation will continue.')


QuickBooks : income=$31,732.00  expenses=$21,534.79  net=$10,197.21
Givebacks  : 20 items  total=$7,761.00
Bank       : beginning=$38,056.59  ending=$48,253.80

History saved -> data/history/April_2026.json


In [15]:
try:
    qb = parse_quickbooks(QB_FILE)
    print('QB OK:', qb['income_total'])
except Exception as e:
    print('QB FAILED:', e)

# Step 2 - test Givebacks alone
try:
    givebacks = parse_givebacks_files(GB_FILE_INFO)
    print('Givebacks OK:', len(givebacks))
except Exception as e:
    print('Givebacks FAILED:', e)

# Step 3 - test Bank alone
try:
    bank = parse_chase_pdf(BANK_FILE)
    print('Bank OK:', bank['beginning_balance'])
except Exception as e:
    print('Bank FAILED:', e)

QB OK: 31732.0
Givebacks OK: 20
Bank OK: 38056.59


## Cell 5 — Build Actuals from History
Assembles the 12-column actuals arrays from all stored monthly JSON files.

In [16]:
def load_all_actuals():
    hist_files = sorted(HISTORY_DIR.glob('*.json'))
    if not hist_files:
        print('No history yet - actuals will be zeros until months are processed.')
        return {}, {}
    income_actuals  = defaultdict(lambda: [0.0]*12)
    expense_actuals = defaultdict(lambda: [0.0]*12)
    for hf in hist_files:
        try:
            entry = json.loads(hf.read_text())
            idx   = entry.get('fiscal_index')
            if idx is None: continue
            for item, val in entry.get('income', {}).items():
                income_actuals[item][idx] = val
            for item, val in entry.get('expenses', {}).items():
                expense_actuals[item][idx] = val
            print(f'  Loaded: {hf.stem:<25} -> {entry["month_label"]:<18} (index {idx}: {FISCAL_MONTHS[idx]})')
        except Exception as e:
            print(f'  Warning: could not load {hf.name}: {e}')
    return dict(income_actuals), dict(expense_actuals)

print('Loading actuals from history...')
INCOME_ACTUALS_LIVE, EXPENSE_ACTUALS_LIVE = load_all_actuals()
print(f'\nIncome items tracked : {len(INCOME_ACTUALS_LIVE)}')
print(f'Expense items tracked: {len(EXPENSE_ACTUALS_LIVE)}')
print(f'\nValues for {MONTH_LABEL} (column {FISCAL_MONTHS[FISCAL_IDX]}):')
income_this_month = {k:v[FISCAL_IDX] for k,v in INCOME_ACTUALS_LIVE.items() if v[FISCAL_IDX]}
expense_this_month = {k:v[FISCAL_IDX] for k,v in EXPENSE_ACTUALS_LIVE.items() if v[FISCAL_IDX]}
print('  Income:')
for k,v in income_this_month.items():  print(f'    {k:<40} ${v:>10,.2f}')
print('  Expenses:')
for k,v in expense_this_month.items(): print(f'    {k:<40} ${v:>10,.2f}')


Loading actuals from history...
  Loaded: April_2026                -> April 2026         (index 9: APR)

Income items tracked : 10
Expense items tracked: 12

Values for April 2026 (column APR):
  Income:
    Basket Dinner                            $ 10,500.00
    Contribution                             $ 14,390.00
    Plant Sale                               $  1,182.00
    Staff Appreciation                       $  1,135.00
    Lawn Signs                               $    855.00
    Skate Night                              $     20.00
    Membership - Teachers                    $     30.00
    Birthday Books-Income                    $     60.00
    Fast Athletics                           $  3,360.00
    Talent Show Income                       $    200.00
  Expenses:
    Bank Charges & Fees                      $     30.50
    Movie Night                              $    668.94
    Arts In Education                        $  4,045.00
    Basket dinner                         

## Cell 6 — Annual Budget Data
*Edit once per fiscal year (July). Last Year = prior year totals.*

In [17]:
# Format: 'Item': (last_year_actual, annual_budget)
INCOME_BUDGET = {
    'Fundraising': {
        'Birthday Books':  (2310.00, 2000.00), 'Book Fair':       (9118.36,  500.00),
        'Croc Charms':     (405.00,   100.00), 'Fall Pictures':   (3350.25, 3000.00),
        'FAST':            (26370.00,1000.00), 'Holiday Boutique':(13417.00,7500.00),
        'Plant Sale':      (8202.68, 8000.00), 'Spiritwear':      (1253.96, 1500.00),
        'Spring Pictures': (0.00,       0.00),
    },
    'Basket Dinner': {
        'Ticket & Raffle Sales': (22165.00, 15000.00),
        'Sponsors':              (7450.00,   4000.00),
    },
    'Membership': {
        'Single':(2580.00,2000.00), 'Family':(1675.00,1000.00),
        'Donations':(290.63,100.00), 'Teachers':(0.00,165.00), 'Student':(0.00,5.00),
    },
    'Program': {
        'Gingerbread U':(3515.00,0.00), 'Staff Appreciation':(1625.00,1000.00),
        'Talent Show':(1070.00,750.00),
    },
    'Grad Class Activities': {
        'Family Contributions':(6480.00,3900.00), 'Lawn Signs':(2560.00,750.00),
        'Treat or Trunk':(1250.00,975.00), 'The Night at Rinx':(0.00,0.00),
    },
}

EXPENSE_BUDGET = {
    'Admin/General': {
        'Accountant':(650.00,650.00), 'Bank Services':(234.94,200.00),
        'Insurance':(350.00,350.00), 'Supplies':(450.22,500.00),
        'Accounting Quickbooks':(410.66,1300.00), 'Training':(68.90,100.00),
        'Website & Remind App':(344.01,1000.00), 'Event Equipment':(563.32,1000.00),
    },
    'Fundraising': {
        'Birthday Books':(503.39,500.00), 'Book Fair':(9523.91,10000.00),
        'Croc Charms':(205.00,0.00), 'Fall Pictures':(0.00,0.00),
        'FAST':(24000.00,17000.00), 'Holiday Boutique':(11714.86,10000.00),
        'Plant Sale':(5615.95,6000.00), 'Spiritwear':(0.00,1000.00),
        'Spring Pictures':(0.00,0.00),
    },
    'Membership': {
        'Council Dues':(125.00,150.00), 'Membership Expenses':(1034.00,1500.00),
    },
    'Basket Dinner': {
        'Entertainment':(1405.42,1000.00), 'Raffles':(1831.70,2000.00),
        'Venue':(8265.20,10000.00),
    },
    'Programs': {
        'Bus Driver Appreciation':(240.00,300.00), 'Electric Parade':(371.89,200.00),
        'Family Connect Nights':(771.00,1500.00), 'Gingerbread U':(4245.66,4500.00),
        'Homecoming':(266.31,150.00), 'K Playdate':(83.13,300.00),
        'K Orientation':(0.00,400.00), 'Milk & Cookies':(358.07,750.00),
        'Multicultural Night':(2022.51,2500.00), 'Outdoor Movie':(1884.20,2500.00),
        'Science Fair':(883.54,1500.00), 'Spring Fling':(0.00,2500.00),
        'Staff Appreciation':(3840.00,4000.00), 'Talent Show':(340.62,1000.00),
        'Talent Show DJ':(550.00,500.00), 'Volunteer Breakfast':(64.00,350.00),
        'Welcome Back Breakfast':(582.66,1000.00), 'WINGO':(0.00,500.00),
    },
    'Donations': {
        'BOE Gifts':(200.00,180.00), 'Cultural Arts':(14651.20,15000.00),
        'Folders':(580.00,600.00), 'Gardening':(0.00,500.00),
        'Hospitality':(124.70,750.00), '5th Staff T-Shirts':(191.78,125.00),
        'Recess Equipment':(1000.00,1000.00), 'School Spirit':(280.76,750.00),
        'Sling Bags':(2354.50,1500.00), 'Spelling Bee':(192.50,225.00),
        'Sunshine Fund':(210.00,500.00), 'Trick or Treat Street':(0.00,250.00),
        'WM Scholarships':(1000.00,1000.00),
    },
    'Grad Class Events': {
        'Monster Bash':(402.84,1300.00), 'Monster Bash DJ':(500.00,500.00),
        'Electric Parade':(0.00,900.00), 'Winter Social':(1167.86,1300.00),
        'Winter Social DJ':(500.00,500.00), 'Moving Up':(715.00,400.00),
        'Picnic':(2441.98,900.00),
    },
    'Grad Class Expenses': {
        'Graduating Class Gifts':(1034.41,500.00), 'Graduating Mural':(244.48,350.00),
        '5th Grade T-Shirts':(1494.25,800.00),
        'Trunk or Treat Fundraiser':(258.67,500.00),
        'The Night at Rinx':(710.00,650.00),
    },
}

def merge_actuals_into_budget(budget_dict, actuals_dict):
    result = {}
    all_budget_items = {item for section in budget_dict.values() for item in section}
    for section, items in budget_dict.items():
        result[section] = {}
        for item, (last_yr, budget) in items.items():
            result[section][item] = (last_yr, budget, actuals_dict.get(item, [0.0]*12))
    unmatched = [k for k in actuals_dict if k not in all_budget_items]
    if unmatched:
        result['Other (from QuickBooks)'] = {}
        print(f'NOTE: {len(unmatched)} QB item(s) not in budget -> added to Other:')
        for item in unmatched:
            print(f'  - {item}')
            result['Other (from QuickBooks)'][item] = (0.0, 0.0, actuals_dict[item])
    return result

INCOME_MERGED  = merge_actuals_into_budget(INCOME_BUDGET,  INCOME_ACTUALS_LIVE)
EXPENSE_MERGED = merge_actuals_into_budget(EXPENSE_BUDGET, EXPENSE_ACTUALS_LIVE)
print(f'Budget + actuals merged  |  Income sections: {len(INCOME_MERGED)}  |  Expense sections: {len(EXPENSE_MERGED)}')


NOTE: 7 QB item(s) not in budget -> added to Other:
  - Basket Dinner
  - Contribution
  - Skate Night
  - Membership - Teachers
  - Birthday Books-Income
  - Fast Athletics
  - Talent Show Income
NOTE: 8 QB item(s) not in budget -> added to Other:
  - Bank Charges & Fees
  - Movie Night
  - Arts In Education
  - Basket dinner
  - Basket Dinner - Venue
  - Welcome Back Staff
  - Scholastic Book Fairs
  - Spring Dance K-5
Budget + actuals merged  |  Income sections: 6  |  Expense sections: 9


## Cell 7 — Excel Style Helpers

In [18]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

NAVY='1F3864'; TEAL='2E75B6'; LTBLUE='BDD7EE'; GOLD='FFD966'
WHITE='FFFFFF'; LGREY='F2F2F2'; GREEN='E2EFDA'; RED_BG='FCE4D6'

SUBHDR_FONT = Font(name='Arial',bold=True,color=WHITE,size=10)
BODY_FONT   = Font(name='Arial',size=10)
BOLD_FONT   = Font(name='Arial',bold=True,size=10)
TOTAL_FONT  = Font(name='Arial',bold=True,size=10,color=NAVY)

NAVY_FILL   = PatternFill('solid',fgColor=NAVY)
TEAL_FILL   = PatternFill('solid',fgColor=TEAL)
LTBLUE_FILL = PatternFill('solid',fgColor=LTBLUE)
GOLD_FILL   = PatternFill('solid',fgColor=GOLD)
LGREY_FILL  = PatternFill('solid',fgColor=LGREY)
GREEN_FILL  = PatternFill('solid',fgColor=GREEN)
RED_FILL    = PatternFill('solid',fgColor=RED_BG)

THIN        = Side(style='thin',   color='AAAAAA')
MED         = Side(style='medium', color=NAVY)
THIN_BORDER = Border(left=THIN,right=THIN,top=THIN,bottom=THIN)
MED_BORDER  = Border(left=MED, right=MED, top=MED, bottom=MED)
MONEY_FMT   = '$#,##0.00_);($#,##0.00)'

def sec_hdr(ws,label,row,n=4):
    ws.merge_cells(f'A{row}:{get_column_letter(n)}{row}')
    c=ws[f'A{row}']; c.value=label
    c.font=Font(name='Arial',bold=True,size=11,color=WHITE); c.fill=NAVY_FILL
    c.alignment=Alignment(horizontal='left',vertical='center',indent=1)
    ws.row_dimensions[row].height=20; return row+1

def col_hdrs(ws,row,labels):
    for i,lbl in enumerate(labels):
        c=ws.cell(row=row,column=i+1,value=lbl)
        c.font=SUBHDR_FONT; c.fill=TEAL_FILL; c.border=THIN_BORDER
        c.alignment=Alignment(horizontal='center' if i>0 else 'left',vertical='center',indent=1 if i==0 else 0)
    ws.row_dimensions[row].height=18; return row+1

def data_row(ws,row,label,amount,shade=False):
    fill=LGREY_FILL if shade else PatternFill()
    ws[f'A{row}'].value=label; ws[f'A{row}'].font=BODY_FONT; ws[f'A{row}'].fill=fill
    ws[f'A{row}'].border=THIN_BORDER; ws[f'A{row}'].alignment=Alignment(indent=2)
    ws[f'B{row}'].value=amount; ws[f'B{row}'].font=BODY_FONT; ws[f'B{row}'].fill=fill
    ws[f'B{row}'].border=THIN_BORDER; ws[f'B{row}'].number_format=MONEY_FMT
    ws[f'B{row}'].alignment=Alignment(horizontal='right')
    for col in ['C','D']: ws[f'{col}{row}'].fill=fill; ws[f'{col}{row}'].border=THIN_BORDER
    ws.row_dimensions[row].height=16

def total_row(ws,row,label,val):
    for col in ['A','B','C','D']:
        ws[f'{col}{row}'].fill=LTBLUE_FILL; ws[f'{col}{row}'].border=MED_BORDER
    ws[f'A{row}'].value=label; ws[f'A{row}'].font=TOTAL_FONT; ws[f'A{row}'].alignment=Alignment(indent=1)
    ws[f'B{row}'].value=val; ws[f'B{row}'].font=TOTAL_FONT
    ws[f'B{row}'].number_format=MONEY_FMT; ws[f'B{row}'].alignment=Alignment(horizontal='right')
    ws.row_dimensions[row].height=18

print('Styles ready')


/usr/local/anaconda3/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Styles ready


## Cell 8 — Sheet Builders

In [19]:
def build_treasurer(ws,qb,bank,month_label,org_name):
    ws.sheet_view.showGridLines=False
    for col,w in zip(['A','B','C','D'],[32,18,18,22]): ws.column_dimensions[col].width=w
    ws.merge_cells('A1:D1'); c=ws['A1']; c.value=org_name.upper()
    c.font=Font(name='Arial',bold=True,size=16,color=NAVY)
    c.alignment=Alignment(horizontal='center',vertical='center'); ws.row_dimensions[1].height=28
    ws.merge_cells('A2:D2'); c=ws['A2']
    c.value=f'Monthly Treasurer Report - {month_label}'
    c.font=Font(name='Arial',bold=True,size=12,color=TEAL); c.alignment=Alignment(horizontal='center')
    ws.merge_cells('A3:D3'); c=ws['A3']
    c.value=f'Generated: {datetime.today().strftime("%B %d, %Y")}'
    c.font=Font(name='Arial',italic=True,size=9,color='888888'); c.alignment=Alignment(horizontal='center')
    row=5; row=sec_hdr(ws,'INCOME',row); row=col_hdrs(ws,row,['Category','Amount','',''])
    inc_s=row
    for i,(k,v) in enumerate(qb['income'].items()): data_row(ws,row,k,v,shade=(i%2==1)); row+=1
    total_row(ws,row,'Total Income',f'=SUM(B{inc_s}:B{row-1})'); it=row; row+=2
    row=sec_hdr(ws,'EXPENSES',row); row=col_hdrs(ws,row,['Category','Amount','',''])
    exp_s=row
    for i,(k,v) in enumerate(qb['expenses'].items()): data_row(ws,row,k,v,shade=(i%2==1)); row+=1
    total_row(ws,row,'Total Expenses',f'=SUM(B{exp_s}:B{row-1})'); et=row; row+=2
    ws.merge_cells(f'A{row}:D{row}'); ws[f'A{row}'].value='NET INCOME / (LOSS)'
    ws[f'A{row}'].font=Font(name='Arial',bold=True,size=11,color=WHITE)
    ws[f'A{row}'].fill=NAVY_FILL; ws[f'A{row}'].alignment=Alignment(horizontal='left',indent=1); row+=1
    for col in ['A','B','C','D']: ws[f'{col}{row}'].fill=GOLD_FILL; ws[f'{col}{row}'].border=MED_BORDER
    ws[f'A{row}'].value='Net Income (Loss)'; ws[f'A{row}'].font=TOTAL_FONT; ws[f'A{row}'].alignment=Alignment(indent=1)
    ws[f'B{row}'].value=f'=B{it}-B{et}'; ws[f'B{row}'].font=TOTAL_FONT
    ws[f'B{row}'].number_format=MONEY_FMT; ws[f'B{row}'].alignment=Alignment(horizontal='right'); row+=2
    
    row=sec_hdr(ws,'BANK RECONCILIATION - Chase Business Checking',row)
    row=col_hdrs(ws,row,['Description','Amount','',''])
    for i,(lbl,amt) in enumerate([
        ('Beginning Balance',           bank['beginning_balance']),
        ('(+) Deposits & Additions',    bank['total_deposits']),
        ('(-) Checks Paid',            -bank['total_checks']),
        ('(-) Electronic Withdrawals', -bank.get('total_withdrawals', 0.0)),   # ← new
        ('(-) Fees',                   -bank['total_fees']),
    ]):
        data_row(ws,row,lbl,amt,shade=(i%2==1)); row+=1
    total_row(ws,row,'Ending Balance (per statement)',bank['ending_balance']); row+=1
    calc = (bank['beginning_balance']
            + bank['total_deposits']
            - bank['total_checks']
            - bank.get('total_withdrawals', 0.0)   # ← new
            - bank['total_fees'])
    diff = calc - bank['ending_balance']
    for lbl,val,fill in [
        ('Calculated Ending Balance', calc, PatternFill()),
        ('Difference (should be $0.00)', diff, GREEN_FILL if abs(diff)<0.01 else RED_FILL)
    ]:
        for col in ['A','B','C','D']: ws[f'{col}{row}'].fill=fill; ws[f'{col}{row}'].border=THIN_BORDER
        ws[f'A{row}'].value=lbl; ws[f'A{row}'].font=BOLD_FONT; ws[f'A{row}'].alignment=Alignment(indent=2)
        ws[f'B{row}'].value=val; ws[f'B{row}'].font=BOLD_FONT
        ws[f'B{row}'].number_format=MONEY_FMT; ws[f'B{row}'].alignment=Alignment(horizontal='right')
        ws.row_dimensions[row].height=16; row+=1
    row+=1
    if bank['daily_balances']:
        row=sec_hdr(ws,'DAILY ENDING BALANCES',row)
        for col,hdr in zip(['A','B'],['Date','Balance']):
            ws[f'{col}{row}'].value=hdr; ws[f'{col}{row}'].font=SUBHDR_FONT
            ws[f'{col}{row}'].fill=TEAL_FILL; ws[f'{col}{row}'].border=THIN_BORDER
            ws[f'{col}{row}'].alignment=Alignment(horizontal='center')
        row+=1
        for i,(dt,bal) in enumerate(sorted(bank['daily_balances'].items())):
            fill=LGREY_FILL if i%2 else PatternFill()
            ws[f'A{row}'].value=dt; ws[f'A{row}'].font=BODY_FONT; ws[f'A{row}'].fill=fill
            ws[f'A{row}'].border=THIN_BORDER; ws[f'A{row}'].alignment=Alignment(horizontal='center')
            ws[f'B{row}'].value=bal; ws[f'B{row}'].font=BODY_FONT; ws[f'B{row}'].fill=fill
            ws[f'B{row}'].border=THIN_BORDER; ws[f'B{row}'].number_format=MONEY_FMT
            ws[f'B{row}'].alignment=Alignment(horizontal='right'); row+=1


def build_budget(ws,title,merged_data,org_name,fiscal_months,current_idx):
    ws.sheet_view.showGridLines=False
    ws.column_dimensions['A'].width=28; ws.column_dimensions['B'].width=13; ws.column_dimensions['C'].width=13
    for i in range(12): ws.column_dimensions[get_column_letter(4+i)].width=9
    ws.column_dimensions[get_column_letter(16)].width=11; ws.column_dimensions[get_column_letter(17)].width=11
    ws.merge_cells(f'A1:{get_column_letter(17)}1'); c=ws['A1']
    c.value=f'{org_name.upper()}  -  {title}'
    c.font=Font(name='Arial',bold=True,size=13,color=WHITE); c.fill=NAVY_FILL
    c.alignment=Alignment(horizontal='center',vertical='center'); ws.row_dimensions[1].height=26
    ws.merge_cells(f'A2:{get_column_letter(17)}2'); c=ws['A2']
    c.value=(f'As of {datetime.today().strftime("%B %d, %Y")}  |  '
             f'Fiscal Year July 2025 - June 2026  |  '
             f'Active month: {fiscal_months[current_idx]}')
    c.font=Font(name='Arial',italic=True,size=9,color='666666'); c.alignment=Alignment(horizontal='center')
    headers=['Category','Last Year','Budget (Annual)']+fiscal_months+['Total','Profit/Loss']
    for ci,hdr in enumerate(headers,1):
        c=ws.cell(row=3,column=ci,value=hdr)
        is_active=(4<=ci<=15 and (ci-4)==current_idx)
        c.font=Font(name='Arial',bold=True,size=9,color=NAVY if is_active else WHITE)
        c.fill=GOLD_FILL if is_active else TEAL_FILL
        c.alignment=Alignment(horizontal='center' if ci>1 else 'left',vertical='center',wrap_text=True)
        c.border=THIN_BORDER
    ws.row_dimensions[3].height=30
    dr=4
    for section,items in merged_data.items():
        ws.merge_cells(f'A{dr}:{get_column_letter(17)}{dr}'); c=ws[f'A{dr}']; c.value=section
        c.font=Font(name='Arial',bold=True,size=10,color=WHITE); c.fill=NAVY_FILL
        c.alignment=Alignment(horizontal='left',vertical='center',indent=1)
        ws.row_dimensions[dr].height=18; ss=dr+1; dr+=1
        for ir,(item,(last_yr,budget,monthly)) in enumerate(items.items()):
            fill=LGREY_FILL if ir%2==1 else PatternFill()
            c=ws.cell(row=dr,column=1,value=item)
            c.font=BODY_FONT; c.fill=fill; c.alignment=Alignment(indent=2); c.border=THIN_BORDER
            for ci,v in [(2,last_yr),(3,budget)]:
                c=ws.cell(row=dr,column=ci,value=v if v else None)
                c.font=BODY_FONT; c.fill=fill; c.number_format=MONEY_FMT
                c.alignment=Alignment(horizontal='right'); c.border=THIN_BORDER
            for mi,v in enumerate(monthly):
                is_active=(mi==current_idx)
                c=ws.cell(row=dr,column=4+mi,value=v if v else None)
                c.font=Font(name='Arial',bold=is_active,size=10)
                c.fill=GOLD_FILL if (is_active and v) else (PatternFill('solid',fgColor='FFF9E6') if is_active else fill)
                c.number_format=MONEY_FMT; c.alignment=Alignment(horizontal='right'); c.border=THIN_BORDER
            c=ws.cell(row=dr,column=16,value=f'=SUM(D{dr}:O{dr})')
            c.font=BODY_FONT; c.fill=fill; c.number_format=MONEY_FMT
            c.alignment=Alignment(horizontal='right'); c.border=THIN_BORDER
            c=ws.cell(row=dr,column=17,value=f'=C{dr}-P{dr}' if budget else None)
            c.font=BODY_FONT; c.fill=fill; c.number_format=MONEY_FMT
            c.alignment=Alignment(horizontal='right'); c.border=THIN_BORDER
            ws.row_dimensions[dr].height=15; dr+=1
        se=dr-1; c=ws.cell(row=dr,column=1,value=f'Total {section}')
        c.font=TOTAL_FONT; c.fill=LTBLUE_FILL; c.alignment=Alignment(indent=1); c.border=MED_BORDER
        for ci in range(2,18):
            cl=get_column_letter(ci)
            c=ws.cell(row=dr,column=ci,value=f'=SUM({cl}{ss}:{cl}{se})')
            c.font=TOTAL_FONT; c.fill=LTBLUE_FILL; c.number_format=MONEY_FMT
            c.alignment=Alignment(horizontal='right'); c.border=MED_BORDER
        ws.row_dimensions[dr].height=18; dr+=2
    c=ws.cell(row=dr,column=1,value='GRAND TOTAL')
    c.font=Font(name='Arial',bold=True,size=11,color=WHITE); c.fill=NAVY_FILL
    c.alignment=Alignment(indent=1); c.border=MED_BORDER
    for ci in range(2,18):
        cl=get_column_letter(ci)
        c=ws.cell(row=dr,column=ci,value=f'=SUMIF(A4:A{dr-1},"Total*",{cl}4:{cl}{dr-1})')
        c.font=Font(name='Arial',bold=True,size=10,color=WHITE); c.fill=NAVY_FILL
        c.number_format=MONEY_FMT; c.alignment=Alignment(horizontal='right'); c.border=MED_BORDER
    ws.row_dimensions[dr].height=20


def build_givebacks(ws,givebacks,bank,org_name):
    ws.sheet_view.showGridLines=False
    for col,w in zip(['A','B','C','D','E','F'],[30,18,12,16,11,25]): ws.column_dimensions[col].width=w
    ws.merge_cells('A1:F1'); c=ws['A1']
    c.value=f'{org_name.upper()}  -  Giveback Reconciliation'
    c.font=Font(name='Arial',bold=True,size=13,color=WHITE); c.fill=NAVY_FILL
    c.alignment=Alignment(horizontal='center',vertical='center'); ws.row_dimensions[1].height=26
    ws.merge_cells('A2:F2'); c=ws['A2']
    c.value=f'Generated: {datetime.today().strftime("%B %d, %Y")}'
    c.font=Font(name='Arial',italic=True,size=9,color='666666'); c.alignment=Alignment(horizontal='center')
    row=4
    for col,hdr in zip(['A','B','C','D','E','F'],['Item','Category','Transactions','Amount','% of Total','Source File']):
        c=ws[f'{col}{row}']; c.value=hdr; c.font=SUBHDR_FONT; c.fill=TEAL_FILL
        c.alignment=Alignment(horizontal='center' if col!='A' else 'left'); c.border=THIN_BORDER
    ws.row_dimensions[row].height=18; row+=1
    ds=row; trn=ds+len(givebacks)
    for i,g in enumerate(givebacks):
        fill=LGREY_FILL if i%2==1 else PatternFill()
        for col in ['A','B','C','D','E','F']: ws[f'{col}{row}'].fill=fill; ws[f'{col}{row}'].border=THIN_BORDER
        ws[f'A{row}'].value=g['item']; ws[f'A{row}'].font=BODY_FONT; ws[f'A{row}'].alignment=Alignment(indent=1)
        ws[f'B{row}'].value=g['category']; ws[f'B{row}'].font=BODY_FONT; ws[f'B{row}'].alignment=Alignment(horizontal='center')
        ws[f'C{row}'].value=g['count']; ws[f'C{row}'].font=BODY_FONT; ws[f'C{row}'].alignment=Alignment(horizontal='center')
        ws[f'D{row}'].value=g['total']; ws[f'D{row}'].font=BODY_FONT
        ws[f'D{row}'].number_format=MONEY_FMT; ws[f'D{row}'].alignment=Alignment(horizontal='right')
        ws[f'E{row}'].value=f'=D{row}/D{trn}'; ws[f'E{row}'].font=BODY_FONT
        ws[f'E{row}'].number_format='0.0%'; ws[f'E{row}'].alignment=Alignment(horizontal='center')
        ws[f'F{row}'].value=g.get('source_file',''); ws[f'F{row}'].font=BODY_FONT; ws[f'F{row}'].alignment=Alignment(indent=1)
        ws.row_dimensions[row].height=15; row+=1
    for col in ['A','B','C','D','E','F']: ws[f'{col}{row}'].fill=LTBLUE_FILL; ws[f'{col}{row}'].border=MED_BORDER
    ws[f'A{row}'].value='TOTAL'; ws[f'A{row}'].font=TOTAL_FONT; ws[f'A{row}'].alignment=Alignment(indent=1)
    ws[f'C{row}'].value=f'=SUM(C{ds}:C{row-1})'; ws[f'C{row}'].font=TOTAL_FONT; ws[f'C{row}'].alignment=Alignment(horizontal='center')
    ws[f'D{row}'].value=f'=SUM(D{ds}:D{row-1})'; ws[f'D{row}'].font=TOTAL_FONT
    ws[f'D{row}'].number_format=MONEY_FMT; ws[f'D{row}'].alignment=Alignment(horizontal='right')
    ws[f'E{row}'].value='100.0%'; ws[f'E{row}'].font=TOTAL_FONT; ws[f'E{row}'].alignment=Alignment(horizontal='center')
    ws.row_dimensions[row].height=18; row+=2
    sec_hdr(ws,'GIVEBACK <-> BANK RECONCILIATION',row,n=6); row+=1
    gb_total=sum(g['total'] for g in givebacks)
    bank_gb=next((d['amount'] for d in bank['deposits']
                  if 'gb payout' in d.get('description','').lower()
                  or 'givebacks' in d.get('description','').lower()),0.0)
    for lbl,val in [('Givebacks Platform Total',gb_total),
                     ('Givebacks Deposit in Bank Statement',bank_gb),
                     ('Difference',gb_total-bank_gb)]:
        is_d=lbl=='Difference'
        fill=(GREEN_FILL if abs(gb_total-bank_gb)<0.01 else RED_FILL) if is_d else PatternFill()
        for col in ['A','B','C','D','E','F']: ws[f'{col}{row}'].fill=fill; ws[f'{col}{row}'].border=THIN_BORDER
        ws[f'A{row}'].value=lbl; ws[f'A{row}'].font=BOLD_FONT; ws[f'A{row}'].alignment=Alignment(indent=2)
        ws[f'D{row}'].value=val; ws[f'D{row}'].font=BOLD_FONT
        ws[f'D{row}'].number_format=MONEY_FMT; ws[f'D{row}'].alignment=Alignment(horizontal='right')
        ws.row_dimensions[row].height=16; row+=1


def build_manifest(ws,gb_folder,qb_folder,bank_folder,org_name,month_label,fiscal_idx,fiscal_months):
    ws.sheet_view.showGridLines=False
    for col,w in zip(['A','B','C','D'],[15,35,22,14]): ws.column_dimensions[col].width=w
    ws.merge_cells('A1:D1'); c=ws['A1']; c.value=f'{org_name.upper()}  -  File Manifest'
    c.font=Font(name='Arial',bold=True,size=13,color=WHITE); c.fill=NAVY_FILL
    c.alignment=Alignment(horizontal='center',vertical='center'); ws.row_dimensions[1].height=26
    ws.merge_cells('A2:D2'); c=ws['A2']
    c.value=(f'Report: {month_label}  |  Column: {fiscal_months[fiscal_idx]}  |  '
             f'Generated: {datetime.today().strftime("%B %d, %Y at %I:%M %p")}')
    c.font=Font(name='Arial',italic=True,size=9,color='666666'); c.alignment=Alignment(horizontal='center')
    row=4
    for col,hdr in zip(['A','B','C','D'],['Type','Filename','Last Modified','Size']):
        c=ws[f'{col}{row}']; c.value=hdr; c.font=SUBHDR_FONT; c.fill=TEAL_FILL
        c.alignment=Alignment(horizontal='left' if col=='B' else 'center'); c.border=THIN_BORDER
    ws.row_dimensions[row].height=18; row+=1
    all_files=([('Givebacks',f) for f in sorted(gb_folder.glob('*.csv'))]+
               [('QuickBooks',f) for f in sorted(qb_folder.glob('*.csv'))]+
               [('Bank Statement',f) for f in sorted(bank_folder.glob('*.pdf'))])
    for i,(ft,fp) in enumerate(all_files):
        fill=LGREY_FILL if i%2==1 else PatternFill()
        stat=fp.stat(); mod=datetime.fromtimestamp(stat.st_mtime).strftime('%Y-%m-%d %H:%M')
        size=f'{stat.st_size/1024:.1f} KB'
        for col in ['A','B','C','D']: ws[f'{col}{row}'].fill=fill; ws[f'{col}{row}'].border=THIN_BORDER
        ws[f'A{row}'].value=ft; ws[f'A{row}'].font=BODY_FONT; ws[f'A{row}'].alignment=Alignment(horizontal='center')
        ws[f'B{row}'].value=fp.name; ws[f'B{row}'].font=BODY_FONT; ws[f'B{row}'].alignment=Alignment(indent=1)
        ws[f'C{row}'].value=mod; ws[f'C{row}'].font=BODY_FONT; ws[f'C{row}'].alignment=Alignment(horizontal='center')
        ws[f'D{row}'].value=size; ws[f'D{row}'].font=BODY_FONT; ws[f'D{row}'].alignment=Alignment(horizontal='right',indent=1)
        ws.row_dimensions[row].height=16; row+=1
    ws.merge_cells(f'A{row}:D{row}')
    ws[f'A{row}'].value=f'Total files processed: {len(all_files)}'
    ws[f'A{row}'].font=BOLD_FONT; ws[f'A{row}'].alignment=Alignment(indent=1)

print('Sheet builders ready')


Sheet builders ready


## Cell 9 — Generate Excel Report

In [20]:
print(f'Building workbook  ->  {ORG_NAME}  |  {MONTH_LABEL}  |  Column: {FISCAL_MONTHS[FISCAL_IDX]}')
wb = openpyxl.Workbook()

ws1 = wb.active; ws1.title = 'Treasurer Report'
build_treasurer(ws1, qb, bank, MONTH_LABEL, ORG_NAME)
print('  Tab 1: Treasurer Report')

ws2 = wb.create_sheet('Income Budget vs Actuals')
build_budget(ws2, 'Budget vs Actuals - Income', INCOME_MERGED, ORG_NAME, FISCAL_MONTHS, FISCAL_IDX)
print(f'  Tab 2: Income Budget vs Actuals  (column {FISCAL_MONTHS[FISCAL_IDX]} updated from {QB_FILE.name})')

ws3 = wb.create_sheet('Expense Budget vs Actuals')
build_budget(ws3, 'Budget vs Actuals - Expenses', EXPENSE_MERGED, ORG_NAME, FISCAL_MONTHS, FISCAL_IDX)
print(f'  Tab 3: Expense Budget vs Actuals  (column {FISCAL_MONTHS[FISCAL_IDX]} updated from {QB_FILE.name})')

ws4 = wb.create_sheet('Giveback Reconciliation')
build_givebacks(ws4, givebacks, bank, ORG_NAME)
print('  Tab 4: Giveback Reconciliation')

ws5 = wb.create_sheet('File Manifest')
build_manifest(ws5, GB_FOLDER, QB_FOLDER, BANK_FOLDER, ORG_NAME, MONTH_LABEL, FISCAL_IDX, FISCAL_MONTHS)
print('  Tab 5: File Manifest')

wb.save(OUTPUT_FILE)
print(f'\nSaved -> {OUTPUT_FILE}')


Building workbook  ->  Setauket School PTA  |  April 2026  |  Column: APR
  Tab 1: Treasurer Report
  Tab 2: Income Budget vs Actuals  (column APR updated from quickbooks_april_2026.csv)
  Tab 3: Expense Budget vs Actuals  (column APR updated from quickbooks_april_2026.csv)
  Tab 4: Giveback Reconciliation
  Tab 5: File Manifest

Saved -> output/Treasurer_Report_April_2026.xlsx


## Cell 10 — Financial Summary

In [21]:
print('='*55)
print(f'  {ORG_NAME}  -  {MONTH_LABEL}')
print(f'  Budget column updated: {FISCAL_MONTHS[FISCAL_IDX]} (index {FISCAL_IDX})')
print('='*55)
print(f'  Income            : ${qb["income_total"]:>12,.2f}')
print(f'  Expenses          : ${qb["expense_total"]:>12,.2f}')
print(f'  Net Income (Loss) : ${qb["net_income"]:>12,.2f}')
print('-'*55)
print(f'  Bank Beginning    : ${bank["beginning_balance"]:>12,.2f}')
print(f'  Bank Ending       : ${bank["ending_balance"]:>12,.2f}')
calc = bank['beginning_balance']+bank['total_deposits']-bank['total_checks']-bank.get('total_withdrawals', 0.0)-bank['total_fees']
diff = calc - bank['ending_balance']
print(f'  Reconciliation    : {"Balanced" if abs(diff)<0.01 else f"Off by ${abs(diff):,.2f}"}')
print(f'  Givebacks Total   : ${sum(g["total"] for g in givebacks):>12,.2f}')
print('='*55)
hist_files = sorted(HISTORY_DIR.glob('*.json'))
print(f'\n  Months stored in history: {len(hist_files)}')
for hf in hist_files:
    e = json.loads(hf.read_text())
    fi = e.get("fiscal_index", "?")
    col = FISCAL_MONTHS[fi] if isinstance(fi, int) else "?"
    print(f'    [{col:<5}] {e["month_label"]:<20} income=${e["income_total"]:>10,.2f}  expenses=${e["expense_total"]:>10,.2f}')
print(f'\n  Output: {OUTPUT_FILE}')


  Setauket School PTA  -  April 2026
  Budget column updated: APR (index 9)
  Income            : $   31,732.00
  Expenses          : $   21,534.79
  Net Income (Loss) : $   10,197.21
-------------------------------------------------------
  Bank Beginning    : $   38,056.59
  Bank Ending       : $   48,253.80
  Reconciliation    : Off by $400.00
  Givebacks Total   : $    7,761.00

  Months stored in history: 1
    [APR  ] April 2026           income=$ 31,732.00  expenses=$ 21,534.79

  Output: output/Treasurer_Report_April_2026.xlsx


## Cell 11 — Push to GitHub
Pushes code changes to your GitHub repository via SSH.

**What gets pushed:** notebook, README, .gitignore, requirements.txt — never input files, output Excel, or `.env`.

**One-time setup (run in terminal, not here):**
```bash
git init
git remote add origin git@github.com:yourname/pta-treasurer.git
# Add your SSH public key at: github.com → Settings → SSH and GPG keys
```


In [ ]:
import subprocess

# Files/folders that should NEVER be pushed
SENSITIVE_PATTERNS = ['input/', 'output/', 'data/', '.env', '*.xlsx', '*.pdf', '*.csv']

def run_git(args, check=True):
    """Run a git command and return (returncode, stdout, stderr)."""
    result = subprocess.run(
        ['git'] + args,
        capture_output=True, text=True
    )
    if check and result.returncode != 0:
        raise RuntimeError(f'git {" ".join(args)} failed:\n{result.stderr}')
    return result.returncode, result.stdout.strip(), result.stderr.strip()

def push_to_github(commit_message=None):
    print('GitHub Push')
    print('='*55)

    # 1. Check git is initialized
    code, _, _ = run_git(['rev-parse', '--git-dir'], check=False)
    if code != 0:
        print('ERROR: Not a git repository.')
        print('Run in terminal:')
        print('  git init')
        print(f'  git remote add {GITHUB_REMOTE} git@github.com:yourname/pta-treasurer.git')
        return

    # 2. Check remote exists
    _, remotes, _ = run_git(['remote'])
    if GITHUB_REMOTE not in remotes.split():
        print(f'ERROR: Remote "{GITHUB_REMOTE}" not found.')
        print(f'Run in terminal: git remote add {GITHUB_REMOTE} git@github.com:yourname/pta-treasurer.git')
        return

    # 3. Check .gitignore exists and protects sensitive files
    gitignore = Path('.gitignore')
    if not gitignore.exists():
        print('Creating .gitignore...')
        gitignore.write_text(
            '# Input files - never commit\n'
            'input/\n'
            'output/\n'
            'data/\n'
            '*.xlsx\n'
            '*.pdf\n'
            '*.csv\n'
            '.env\n'
            '__pycache__/\n'
            '*.pyc\n'
            '.DS_Store\n'
        )
        print('  .gitignore created')

    # 4. Show git status
    _, status, _ = run_git(['status', '--short'])
    if not status:
        print('Nothing to commit - working tree clean.')
        return

    print('\nFiles changed:')
    for line in status.split('\n'):
        flag = line[:2].strip()
        fname = line[3:]
        # Warn if sensitive file would be staged
        is_sensitive = any(
            fname.startswith(p.rstrip('*').rstrip('/')) or
            fname.endswith(p.lstrip('*'))
            for p in SENSITIVE_PATTERNS
        )
        icon = '  SKIP (sensitive)' if is_sensitive else '  will push'
        print(f'  {flag} {fname:<45} {icon}')

    # 5. Ask for commit message
    if commit_message is None:
        default_msg = f'Update report code - {MONTH_LABEL}'
        commit_message = input(f'\nCommit message [{default_msg}]: ').strip()
        if not commit_message:
            commit_message = default_msg

    # 6. Stage only safe files (not sensitive paths)
    print('\nStaging files...')
    safe_to_add = []
    for line in status.split('\n'):
        fname = line[3:].strip()
        is_sensitive = any(
            fname.startswith(p.rstrip('*').rstrip('/')) or
            fname.endswith(p.lstrip('*'))
            for p in SENSITIVE_PATTERNS
        )
        if not is_sensitive and fname:
            safe_to_add.append(fname)

    if not safe_to_add:
        print('No safe files to push (all changes are in sensitive paths).')
        return

    for f in safe_to_add:
        run_git(['add', f])
        print(f'  staged: {f}')

    # 7. Commit
    print(f'\nCommitting: "{commit_message}"')
    run_git(['commit', '-m', commit_message])

    # 8. Push
    print(f'Pushing to {GITHUB_REMOTE}/{GITHUB_BRANCH}...')
    run_git(['push', GITHUB_REMOTE, GITHUB_BRANCH])

    # 9. Show result
    _, log, _ = run_git(['log', '--oneline', '-1'])
    _, remote_url, _ = run_git(['remote', 'get-url', GITHUB_REMOTE])
    repo_url = remote_url.replace('git@github.com:', 'https://github.com/').replace('.git','')

    print('='*55)
    print(f'  Pushed successfully!')
    print(f'  Commit : {log}')
    print(f'  Repo   : {repo_url}')
    print('='*55)


# ── Run ───────────────────────────────────────────────────────────────────────
push_to_github()


GitHub Push
Creating .gitignore...
  .gitignore created

Files changed:
  ?? .gitignore                                      will push
  ?? .ipynb_checkpoints/                             will push
  ?? PTA_Treasurer_Report.ipynb                      will push
  ?? PTA_Treasurer_Report_v4.ipynb                   will push
  ?? README.md                                       will push
  ?? data                                            SKIP (sensitive)
  ?? files                                           will push
  ?? generate_report.py                              will push
  ?? requirements.txt                                will push
